# 02 — Reproduce every figure

Every chart the project publishes, recomputed from `tier1_findings.parquet` and the discovery
tables. **The numbers behind each chart are printed immediately above it** so you can check the
figure against the data rather than trusting it.

Runs in Colab with no setup.

> ## Scope — read this first
>
> **This is Tier 1: POTS / dysautonomia literature ONLY** — chunks 0–75 of a 286-chunk corpus,
> **~3,121 of ~14,263 corpus papers**.
>
> ME/CFS, fibromyalgia, EDS/hypermobility, MCAS, IBS, migraine and autism appear here **only as
> incidental mentions inside POTS/dysautonomia papers**. Those conditions' own literatures are in
> Tiers 2–4 and are **not yet extracted**. This is **not** a cross-condition resource.
>
> **Not medical advice.** Machine-extracted research information, not clinical guidance. No
> diagnostic or treatment recommendations. Personal health questions belong with a qualified
> clinician.
>
> **Attribution.** Every finding links to its source paper with authors and journal. These are
> automated summaries of other researchers' work — the credit is theirs, and the original paper is
> the authority. `provenance_quote_trunc` is short verbatim publisher-copyright text included for
> verification only; always keep it next to its citation and never bulk-export it.

In [1]:
# ── Data location ──────────────────────────────────────────────────────────────
# Repoint this constant at your own fork if you want. Raw GitHub is used in Colab;
# a local ./data or ../data directory (i.e. running inside a clone) wins if present.
RAW_BASE = "https://raw.githubusercontent.com/USER/REPO/main/data"

import os, io, json, urllib.request
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt

_LOCAL = next((p for p in ("data", "../data", "../../data") if os.path.isdir(p)), None)
_CACHE = ".cache_tier1"; os.makedirs(_CACHE, exist_ok=True)

def data_path(name):
    """Return a readable path for a data file: local clone if present, else download once."""
    if _LOCAL and os.path.exists(os.path.join(_LOCAL, name)):
        return os.path.join(_LOCAL, name)
    dst = os.path.join(_CACHE, name)
    if not os.path.exists(dst):
        url = f"{RAW_BASE}/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, dst)
    return dst

def load(name):
    p = data_path(name)
    if p.endswith(".parquet"): return pd.read_parquet(p)
    if p.endswith(".json"):    return json.load(open(p))
    return pd.read_csv(p, low_memory=False)

print("data source:", _LOCAL if _LOCAL else RAW_BASE)

data source: ../data


In [2]:
F  = load("tier1_findings.parquet")
P  = load("paper_metadata.parquet")
A  = load("association_table.parquet")
U  = load("understudied_signals.parquet")
C  = load("contradictions.parquet")
NL = load("null_results.parquet")
CV = load("convergence_tier1.parquet")
V  = load("entity_vocabulary.parquet")

def show(title, d):
    print(f"--- {title} ---")
    for k, v in (d.items() if isinstance(d, dict) else d):
        k = str(k)
        print(f"  {k:44s} {v:>10,}" if isinstance(v, (int, float, np.integer, np.floating)) else f"  {k:44s} {v}")
    print()

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 9, "figure.dpi": 110})
print("loaded:", {k: len(v) for k, v in
      dict(findings=F, papers=P, assoc=A, understudied=U, contradictions=C, nulls=NL, convergence=CV, vocab=V).items()})

loaded: {'findings': 36459, 'papers': 3129, 'assoc': 555, 'understudied': 329, 'contradictions': 532, 'nulls': 5324, 'convergence': 308, 'vocab': 2019}


## Figure 1 — Corpus scope

The first figure has to state the limit, not hide it: Tier 1 is a slice of the corpus.

In [3]:
covered, corpus_total = F.canonical_pmid.nunique(), 14263
show("Fig 1 — scope", dict(papers_covered_tier1=covered, corpus_papers_total=corpus_total,
     papers_not_yet_extracted=corpus_total-covered, chunks_tier1=76, chunks_total=286,
     findings=len(F), distinct_finding_text=F.finding_text.nunique()))
print(f"  coverage: {100*covered/corpus_total:.1f}% of corpus papers")

--- Fig 1 — scope ---
  papers_covered_tier1                              3,121
  corpus_papers_total                              14,263
  papers_not_yet_extracted                         11,142
  chunks_tier1                                         76
  chunks_total                                        286
  findings                                         36,459
  distinct_finding_text                            36,032

  coverage: 21.9% of corpus papers


In [4]:
fig, ax = plt.subplots(figsize=(8.6, 2.3))
ax.barh([0], [covered], color="#4C72B0")
ax.barh([0], [corpus_total-covered], left=[covered], color="#D9D9D9")
ax.text(covered/2, 0, f"Tier 1 extracted\n{covered:,} papers", ha="center", va="center",
        color="white", fontsize=8.5, fontweight="bold")
ax.text(covered + (corpus_total-covered)/2, 0, f"Tiers 2–4 NOT yet extracted\n{corpus_total-covered:,} papers",
        ha="center", va="center", color="#333333", fontsize=8.5)
ax.set_yticks([]); ax.set_ylim(-0.55, 0.55); ax.set_xlim(0, corpus_total)
ax.set_xlabel("papers in corpus")
ax.set_title(f"Tier 1 covers the POTS/dysautonomia core: {covered:,} of {corpus_total:,} corpus papers "
             f"({100*covered/corpus_total:.0f}%)", loc="left", pad=10)
fig.tight_layout(); fig.savefig("fig01_scope.png", bbox_inches="tight")

## Figure 2 — Findings and papers by condition label

**This is the figure most likely to be misread.** Non-POTS bars count POTS/dysautonomia papers
that *mention* the condition. They are not those conditions' literatures.

In [5]:
cond = pd.DataFrame({"findings": F.condition.value_counts(),
                     "papers": F.groupby("condition").canonical_pmid.nunique()}).sort_values("findings")
show("Fig 2 — by condition (findings, papers)", [(i, f"{r.findings:,} findings / {r.papers:,} papers")
                                                 for i, r in cond[::-1].iterrows()])

--- Fig 2 — by condition (findings, papers) ---
  dysautonomia                                 18,293 findings / 2,332 papers
  general                                      5,854 findings / 853 papers
  POTS                                         5,078 findings / 749 papers
  long_COVID                                   2,608 findings / 324 papers
  multiple                                     1,129 findings / 345 papers
  ME_CFS                                       845 findings / 136 papers
  fibromyalgia                                 756 findings / 103 papers
  small_fiber_neuropathy                       642 findings / 202 papers
  EDS_hypermobility                            625 findings / 146 papers
  migraine                                     255 findings / 61 papers
  MCAS                                         236 findings / 90 papers
  IBS_gut                                      90 findings / 64 papers
  autism                                       44 findings / 18 pap

In [6]:
POTS_CORE = {"POTS","dysautonomia","general","multiple","long_COVID","orthostatic_hypotension","small_fiber_neuropathy"}
colors = ["#4C72B0" if i in POTS_CORE else "#DD8452" for i in cond.index]
fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.barh(range(len(cond)), cond.findings.values, color=colors)
for i, (n, pp) in enumerate(zip(cond.findings.values, cond.papers.values)):
    ax.text(n + 250, i, f"{n:,} ({pp:,} papers)", va="center", fontsize=7.5)
ax.set_yticks(range(len(cond))); ax.set_yticklabels(cond.index, fontsize=8)
ax.set_xlim(0, cond.findings.max()*1.28); ax.set_xlabel("findings")
ax.set_title("Findings by condition label — orange bars are INCIDENTAL MENTIONS\ninside POTS/dysautonomia papers, not those conditions' literature", loc="left")
fig.tight_layout(); fig.savefig("fig02_conditions.png")

## Figure 3 — Evidence quality profile

Study design tier, publication stratum, certainty, and whether the source claims causation.

In [7]:
show("Fig 3a — evidence_tier", F.evidence_tier.value_counts().to_dict())
show("Fig 3b — certainty", F.certainty.value_counts().to_dict())
show("Fig 3c — causal_or_correlational", F.causal_or_correlational.value_counts().to_dict())
show("Fig 3d — evidence_stratum", F.evidence_stratum.value_counts().to_dict())
print(f"  RCT+meta share: {100*F.evidence_tier.isin(['RCT','meta-analysis']).mean():.1f}%")
print(f"  hedged+mixed certainty share: {100*F.certainty.isin(['hedged','mixed']).mean():.1f}%")

--- Fig 3a — evidence_tier ---
  cohort                                            8,239
  cross-sectional                                   7,009
  review/expert-opinion                             6,507
  case-report                                       3,777
  case-series                                       3,239
  case-control                                      3,184
  mechanistic/in-vitro                              2,787
  RCT                                                 877
  meta-analysis                                       840

--- Fig 3b — certainty ---
  confirmed                                        23,310
  hedged                                            9,347
  mixed                                             3,802

--- Fig 3c — causal_or_correlational ---
  correlational-only                               23,193
  not-stated                                        8,893
  causal-claim                                      4,373

--- Fig 3d — evidence_stratu

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.4))
for ax, (col, ttl) in zip(axes.ravel(), [("evidence_tier","Study design tier"),
        ("evidence_stratum","Publication stratum"), ("certainty","How firmly stated"),
        ("causal_or_correlational","Causal claim?")]):
    vc = F[col].value_counts()[::-1]
    ax.barh(range(len(vc)), vc.values, color="#55A868")
    for i, n in enumerate(vc.values): ax.text(n + F.shape[0]*0.006, i, f"{n:,}", va="center", fontsize=7)
    ax.set_yticks(range(len(vc))); ax.set_yticklabels(vc.index, fontsize=7.5)
    ax.set_xlim(0, vc.max()*1.22); ax.set_title(ttl, loc="left"); ax.set_xlabel("findings")
fig.suptitle("Most findings are observational, correlational, and hedged", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); fig.savefig("fig03_evidence.png")

## Figure 4 — Entity coverage

**The number that matters is 29.8%, not 53.0%.** `entity` (the original string) and
`canonical_entity` (the mapped rollup key) are different columns with different coverage, and the
gap between them has two separate causes: rows that name no entity at all (by design), and rows
whose entity string was never canonicalized (head-only mapping). The first panel below breaks the
whole table into those four mutually exclusive groups so the difference is impossible to miss.

In [9]:
cov = dict(total_rows=len(F),
    rows_with_entity_string=int(F.entity.notna().sum()),
    rows_with_canonical_entity=int(F.canonical_entity.notna().sum()),
    rows_with_ontology_id=int(F.canonical_ontology_id.notna().sum()),
    distinct_canonical_entities=int(F.canonical_entity.nunique()),
    distinct_ontology_ids=int(F.canonical_ontology_id.nunique()))
_e = F.entity.notna() & (F.entity.astype(str).str.strip() != "")
_c = F.canonical_entity.notna() & (F.canonical_entity.astype(str).str.strip() != "")
cov["rows_entity_but_no_canonical"] = int((_e & ~_c).sum())
cov["rows_no_entity_at_all"] = int((~_e).sum())
cov["papers_with_canonical_entity"] = int(F.loc[_c, "canonical_pmid"].nunique())
cov["papers_total"] = int(F.canonical_pmid.nunique())
show("Fig 4 — entity coverage", cov)
print(f"  entity string      : {100*_e.mean():.1f}% of rows")
print(f"  canonical_entity   : {100*_c.mean():.1f}% of rows  <- USE THIS for entity views")
print(f"  the 23.2-pt gap    : {int((_e & ~_c).sum()):,} rows carry a string with no canonical mapping")
print(f"  papers covered     : {cov['papers_with_canonical_entity']:,} of {cov['papers_total']:,}")
_tail = F.loc[_e & ~_c, "entity_norm_original"].value_counts()
print(f"  unmapped strings   : {F.loc[_e & ~_c,'entity_norm_original'].nunique():,} distinct "
      f"({int((_tail==1).sum()):,} singletons, max {int(_tail.max())} rows) - a genuine tail")
show("Fig 4b — map_confidence (rows)", F.map_confidence.value_counts().to_dict())
show("Fig 4c — vocabulary typed entries", V[V.entity_kind.notna()].map_confidence.value_counts().to_dict())
show("Fig 4d — ontology source (rows)", F.canonical_ontology_source.value_counts().to_dict())

--- Fig 4 — entity coverage ---
  total_rows                                       36,459
  rows_with_entity_string                          19,307
  rows_with_canonical_entity                       10,858
  rows_with_ontology_id                             4,321
  distinct_canonical_entities                         863
  distinct_ontology_ids                               368
  rows_entity_but_no_canonical                      8,449
  rows_no_entity_at_all                            17,152
  papers_with_canonical_entity                      2,185
  papers_total                                      3,121

  entity string      : 53.0% of rows
  canonical_entity   : 29.8% of rows  <- USE THIS for entity views
  the 23.2-pt gap    : 8,449 rows carry a string with no canonical mapping
  papers covered     : 2,185 of 3,121
  unmapped strings   : 7,605 distinct (6,761 singletons, max 2 rows) - a genuine tail
--- Fig 4b — map_confidence (rows) ---
  high                                       

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
labels = ["canonical\nentity", "string but NO\ncanonical", "no entity\nat all", "ontology\nID"]
vals = [cov["rows_with_canonical_entity"], cov["rows_entity_but_no_canonical"],
        cov["rows_no_entity_at_all"], cov["rows_with_ontology_id"]]
axes[0].bar(labels, vals, color=["#4C72B0","#DD8452","#BBBBBB","#2F4B7C"])
axes[0].axhline(len(F), color="#C44E52", ls="--", lw=1)
axes[0].text(3.45, len(F), f"all rows {len(F):,}", color="#C44E52", fontsize=7, va="bottom", ha="right")
for i, n in enumerate(vals): axes[0].text(i, n + 600, f"{n:,}\n{100*n/len(F):.1f}%", ha="center", fontsize=7)
axes[0].set_ylim(0, len(F)*1.20); axes[0].set_ylabel("findings")
axes[0].tick_params(axis="x", labelsize=7.5)
axes[0].set_title(f"Entity views cover {100*cov['rows_with_canonical_entity']/len(F):.1f}% of findings, not "
                  f"{100*cov['rows_with_entity_string']/len(F):.1f}%\n"
                  "(first three bars are mutually exclusive)", loc="left", fontsize=9)
mc = F.map_confidence.value_counts().reindex(["high","medium","low"])
axes[1].bar(mc.index, mc.values, color=["#55A868","#DD8452","#C44E52"])
for i, n in enumerate(mc.values): axes[1].text(i, n + 120, f"{n:,}", ha="center", fontsize=7.5)
axes[1].set_title("Mapping confidence (rows)\nfilter to 'high' for load-bearing work", loc="left")
axes[1].set_ylabel("findings"); axes[1].set_ylim(0, mc.max()*1.15)
ek = F.entity_kind.value_counts()[::-1]
axes[2].barh(range(len(ek)), ek.values, color="#4C72B0")
for i, n in enumerate(ek.values): axes[2].text(n + 40, i, f"{n:,}", va="center", fontsize=7.5)
axes[2].set_yticks(range(len(ek))); axes[2].set_yticklabels(ek.index, fontsize=8)
axes[2].set_xlim(0, ek.max()*1.2); axes[2].set_title("Entity kinds", loc="left"); axes[2].set_xlabel("findings")
fig.tight_layout(); fig.savefig("fig04_entity_coverage.png")

## Figure 5 — Top entities by distinct papers

From the **29.8% of rows carrying a `canonical_entity`** — 2,185 of 3,121 papers, not the whole
table. `papers_direct` excludes `split_derived` rows, which is the number to quote when a paper
count is the claim. Note also that these counts use the `canonical_pmid` basis; the raw-`pmid`
basis gives slightly different numbers for 3 of 863 entities (elp1 58 vs 53, norepinephrine 195
vs 194, ivabradine 63 vs 62) because some merged groups were preprint→published pairs.

In [11]:
ent = F[F.canonical_entity.notna()].copy()
ent["_direct_pmid"] = ent.canonical_pmid.where(~ent.split_derived)
top = (ent.groupby(["canonical_entity","entity_kind"])
          .agg(papers=("canonical_pmid","nunique"),
               papers_direct=("_direct_pmid","nunique"),
               findings=("finding_text","size"))
          .sort_values("papers", ascending=False))
show("Fig 5 — top 20 entities (papers / papers_direct / findings)",
     [(f"{i[0]} [{i[1]}]", f"{int(r.papers)} / {int(r.papers_direct)} / {int(r.findings)}")
      for i, r in top.head(20).iterrows()])

--- Fig 5 — top 20 entities (papers / papers_direct / findings) ---
  norepinephrine [molecule-or-analyte]         194 / 176 / 270
  heart rate [physiological-measure]           178 / 168 / 271
  heart rate variability [physiological-measure] 170 / 168 / 291
  orthostatic hypotension [clinical-sign-or-condition] 115 / 115 / 249
  midodrine [drug-or-drug-class]               95 / 86 / 153
  sdnn [physiological-measure]                 94 / 80 / 162
  rmssd [physiological-measure]                84 / 70 / 121
  lf/hf ratio [physiological-measure]          81 / 78 / 117
  snca [gene-or-protein]                       77 / 77 / 182
  il-6 [molecule-or-analyte]                   75 / 51 / 92
  intravenous immunoglobulin [drug-or-drug-class] 62 / 62 / 84
  ivabradine [drug-or-drug-class]              62 / 60 / 125
  systolic blood pressure [physiological-measure] 62 / 55 / 89
  fludrocortisone [drug-or-drug-class]         61 / 54 / 72
  blood pressure [physiological-measure]       57 / 51 / 7

In [12]:
t = top.head(18)[::-1]
KIND_C = {"physiological-measure":"#4C72B0","molecule-or-analyte":"#55A868","drug-or-drug-class":"#DD8452",
          "gene-or-protein":"#8172B3","clinical-sign-or-condition":"#C44E52","other":"#937860"}
fig, ax = plt.subplots(figsize=(8.4, 5.6))
names = [i[0] for i in t.index]; kinds = [i[1] for i in t.index]
ax.barh(range(len(t)), t.papers.values, color=[KIND_C[k] for k in kinds])
for i, (a, b) in enumerate(zip(t.papers.values, t.papers_direct.values)):
    ax.text(a + 2, i, f"{int(a)}" + (f" ({int(b)} direct)" if b != a else ""), va="center", fontsize=7.5)
ax.set_yticks(range(len(t))); ax.set_yticklabels(names, fontsize=8)
ax.set_xlim(0, t.papers.max()*1.25); ax.set_xlabel("distinct papers (canonical_pmid)")
ax.set_title(f"Most-studied canonical entities in Tier 1\n"
             f"(canonical_entity rows only: {len(ent):,} of {len(F):,} findings = "
             f"{100*len(ent)/len(F):.1f}%, in {ent.canonical_pmid.nunique():,} of {F.canonical_pmid.nunique():,} papers)",
             loc="left", fontsize=9)
ax.legend(handles=[plt.Rectangle((0,0),1,1,color=c,label=k) for k,c in KIND_C.items() if k in set(kinds)],
          frameon=False, fontsize=7.5, loc="lower right")
fig.tight_layout(); fig.savefig("fig05_top_entities.png")

## Figure 6 — Direction of reported effects

`not applicable` and `no significant difference` together are a large share of rows. Any
direction-agreement metric must exclude them rather than count them as agreement.

In [13]:
show("Fig 6 — direction", F.direction.value_counts().to_dict())
testable = F[~F.direction.isin(["not applicable","no significant difference"])]
print(f"  directional (testable) rows: {len(testable):,} ({100*len(testable)/len(F):.1f}%)")
print(f"  non-directional rows        : {len(F)-len(testable):,}")
show("Fig 6b — relation_type", F.relation_type.value_counts().to_dict())

--- Fig 6 — direction ---
  elevated                                         11,761
  not applicable                                    9,277
  depleted/reduced                                  5,763
  dysregulated (unspecified direction)              5,126
  no significant difference                         4,481
  sex-difference-reported                              50
  mixed                                                 1

  directional (testable) rows: 22,701 (62.3%)
  non-directional rows        : 13,758
--- Fig 6b — relation_type ---
  ASSOCIATED_WITH                                  15,738
  TARGETED_BY_TREATMENT                             5,897
  NO_ASSOCIATION                                    3,832
  DIAGNOSED_BY                                      1,613
  PREDICTS                                          1,579
  BIOMARKER_FOR                                     1,538
  DOWNREGULATED_IN                                  1,477
  CO_OCCURS_WITH                             

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
dv = F.direction.value_counts()[::-1]
axes[0].barh(range(len(dv)), dv.values,
             color=["#BBBBBB" if k in ("not applicable","no significant difference") else "#4C72B0" for k in dv.index])
for i, n in enumerate(dv.values): axes[0].text(n + 150, i, f"{n:,}", va="center", fontsize=7.5)
axes[0].set_yticks(range(len(dv))); axes[0].set_yticklabels(dv.index, fontsize=8)
axes[0].set_xlim(0, dv.max()*1.2); axes[0].set_xlabel("findings")
axes[0].set_title("Direction of effect — grey bars are NOT directional\nand must be excluded from agreement metrics", loc="left")
rv = F.relation_type.value_counts()[::-1]
axes[1].barh(range(len(rv)), rv.values, color=["#C44E52" if k == "NO_ASSOCIATION" else "#55A868" for k in rv.index])
for i, n in enumerate(rv.values): axes[1].text(n + 200, i, f"{n:,}", va="center", fontsize=7.5)
axes[1].set_yticks(range(len(rv))); axes[1].set_yticklabels(rv.index, fontsize=7.5)
axes[1].set_xlim(0, rv.max()*1.2); axes[1].set_xlabel("findings")
axes[1].set_title("Relation types — NO_ASSOCIATION (red) is a real\nfinding, not a missing value", loc="left")
fig.tight_layout(); fig.savefig("fig06_direction.png")

## Figure 7 — The discovery tables at a glance

In [15]:
cross = (A.included_in_primary & A.significant_fdr05 & (A.entity_kind_a != A.entity_kind_b)
         & ~A.trivial_component_link & ~A.name_level_nested & ~A.surface_string_linked)
disc = dict(
    associations_total=len(A), associations_tested=int(A.included_in_primary.sum()),
    associations_significant_q05=int(A.loc[A.included_in_primary,"significant_fdr05"].sum()),
    associations_crosskind_nontrivial=int(cross.sum()),
    understudied_total=len(U), understudied_headline_eligible=int(U.headline_eligible.sum()),
    understudied_replication_targets=int((U.headline_eligible & (U.n_distinct_last_authors>=2) & ~U.case_report_only).sum()),
    contradictions_total=len(C),
    contradictions_direction_flip=int((C.conflict_type=="hard-direction-flip").sum()),
    contradictions_null_vs_positive=int((C.conflict_type=="null-vs-positive").sum()),
    contradictions_visible_moderator=int(C.has_visible_moderator.sum()),
    contradictions_broad_condition_flag=int(C.broad_condition_flag.sum()),
    nulls_total=len(NL), convergence_total=len(CV),
    convergence_passes_nonpots_bar=int(CV.passes_nonpots_bar.sum()),
    convergence_headline_eligible=int(CV.headline_eligible.sum()))
show("Fig 7 — discovery tables", disc)
show("Fig 7b — null_results by class", NL.null_class.value_counts().to_dict())
print("  treatment failures at RCT/meta tier:",
      int(((NL.null_class.str.contains("treatment")) & NL.evidence_tier.isin(["RCT","meta-analysis"])).sum()))

--- Fig 7 — discovery tables ---
  associations_total                                  555
  associations_tested                                 547
  associations_significant_q05                        456
  associations_crosskind_nontrivial                   101
  understudied_total                                  329
  understudied_headline_eligible                      226
  understudied_replication_targets                    198
  contradictions_total                                532
  contradictions_direction_flip                       268
  contradictions_null_vs_positive                     264
  contradictions_visible_moderator                    127
  contradictions_broad_condition_flag                 376
  nulls_total                                       5,324
  convergence_total                                   308
  convergence_passes_nonpots_bar                      106
  convergence_headline_eligible                        45

--- Fig 7b — null_results by class ---

In [16]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
def funnel(ax, pairs, title, color):
    labels = [p[0] for p in pairs][::-1]; vals = [p[1] for p in pairs][::-1]
    ax.barh(range(len(vals)), vals, color=color)
    for i, n in enumerate(vals): ax.text(n + max(vals)*0.03, i, f"{n:,}", va="center", fontsize=7.5)
    ax.set_yticks(range(len(vals))); ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlim(0, max(vals)*1.3); ax.set_title(title, loc="left")
funnel(axes[0], [("all pairs", len(A)), ("tested (non-trivial)", disc["associations_tested"]),
                 ("significant q<0.05", disc["associations_significant_q05"]),
                 ("cross-kind, non-trivial", disc["associations_crosskind_nontrivial"])],
       "Associations: filtering funnel", "#4C72B0")
funnel(axes[1], [("candidates", len(U)), ("headline-eligible", disc["understudied_headline_eligible"]),
                 ("+ >=2 indep. last authors\n  & not case-report-only", disc["understudied_replication_targets"])],
       "Under-studied signals", "#55A868")
funnel(axes[2], [("all conflicts", len(C)), ("direction flips", disc["contradictions_direction_flip"]),
                 ("null-vs-positive", disc["contradictions_null_vs_positive"]),
                 ("visible moderator", disc["contradictions_visible_moderator"]),
                 ("broad-condition caveat", disc["contradictions_broad_condition_flag"])],
       "Contradictions", "#DD8452")
funnel(axes[3], [("all records", len(CV)), (">=2 non-POTS papers", disc["convergence_passes_nonpots_bar"]),
                 ("headline-eligible", disc["convergence_headline_eligible"])],
       "Convergence (non-POTS = incidental)", "#8172B3")
fig.tight_layout(); fig.savefig("fig07_discovery_tables.png")

## Figure 8 — Null and failed results

Negatives were extracted deliberately rather than dropped. The RCT/meta-tier treatment failures
are the most decision-relevant subset in the repository.

In [17]:
show("Fig 8 — null_class", NL.null_class.value_counts().to_dict())
show("Fig 8b — null tier profile", NL.evidence_tier.value_counts().to_dict())
tf = NL[NL.null_class.str.contains("treatment")]
print(f"  treatment failures: {len(tf):,} | at RCT/meta tier: {int(tf.evidence_tier.isin(['RCT','meta-analysis']).sum())}")
print()
print("--- Fig 8c — negative_strength is a CONTINUOUS score, not a category ---")
print(NL.negative_strength.describe().round(2).to_string())
print("  higher = the negative was stated more firmly")

--- Fig 8 — null_class ---
  biomarker/mechanism null                          3,076
  treatment/intervention failure                    1,340
  diagnostic null                                     908

--- Fig 8b — null tier profile ---
  cohort                                            1,407
  cross-sectional                                   1,206
  case-control                                        622
  case-report                                         580
  case-series                                         408
  review/expert-opinion                               382
  mechanistic/in-vitro                                308
  RCT                                                 275
  meta-analysis                                       136

  treatment failures: 1,340 | at RCT/meta tier: 315

--- Fig 8c — negative_strength is a CONTINUOUS score, not a category ---
count    5324.00
mean        1.89
std         1.02
min         0.36
25%         1.20
50%         2.00
75%         

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
nc = NL.null_class.value_counts()[::-1]
axes[0].barh(range(len(nc)), nc.values, color="#C44E52")
for i, n in enumerate(nc.values): axes[0].text(n + 60, i, f"{n:,}", va="center", fontsize=8)
axes[0].set_yticks(range(len(nc))); axes[0].set_yticklabels(nc.index, fontsize=8)
axes[0].set_xlim(0, nc.max()*1.2); axes[0].set_title("Null and failed results by class", loc="left"); axes[0].set_xlabel("findings")
tv = tf.evidence_tier.value_counts()[::-1]
axes[1].barh(range(len(tv)), tv.values, color=["#C44E52" if k in ("RCT","meta-analysis") else "#BBBBBB" for k in tv.index])
for i, n in enumerate(tv.values): axes[1].text(n + 5, i, f"{n:,}", va="center", fontsize=8)
axes[1].set_yticks(range(len(tv))); axes[1].set_yticklabels(tv.index, fontsize=8)
axes[1].set_xlim(0, tv.max()*1.25)
axes[1].set_title("Treatment failures by design tier\n(red = RCT/meta, the strongest negatives)", loc="left")
axes[2].hist(NL.negative_strength.dropna(), bins=20, color="#937860")
axes[2].axvline(NL.negative_strength.median(), color="#C44E52", ls="--", lw=1.2)
axes[2].text(NL.negative_strength.median() + 0.12, axes[2].get_ylim()[1]*0.98,
             f"median {NL.negative_strength.median():.2f}", color="#C44E52", fontsize=7.5,
             va="top", ha="left")
axes[2].set_ylim(0, axes[2].get_ylim()[1]*1.12)
axes[2].set_xlabel("negative_strength (higher = stated more firmly)"); axes[2].set_ylabel("findings")
axes[2].set_title("How firmly the negative was stated\n(continuous score, not a category)", loc="left")
fig.tight_layout(); fig.savefig("fig08_nulls.png")

## Figure 9 — Corpus growth by year

Recency skew here is corpus-construction bias, not a real surge in research volume.

In [19]:
py = P.groupby("year").canonical_pmid.nunique()
fy = F.year.value_counts().sort_index()
show("Fig 9 — papers per year", {str(int(k)): int(v) for k, v in py.items()})
show("Fig 9b — findings per year", {str(int(k)): int(v) for k, v in fy.items()})

--- Fig 9 — papers per year ---
  2019                                                  1
  2020                                                 20
  2021                                                253
  2022                                                315
  2023                                                403
  2024                                                495
  2025                                                896
  2026                                                740

--- Fig 9b — findings per year ---
  2019                                                 19
  2020                                                185
  2021                                              2,781
  2022                                              3,688
  2023                                              4,675
  2024                                              5,773
  2025                                             11,324
  2026                                              8,009



In [20]:
fig, ax = plt.subplots(figsize=(7.6, 3.8))
ax.bar(fy.index.astype(int).astype(str), fy.values, color="#4C72B0", label="findings")
ax2 = ax.twinx()
ax2.plot(range(len(py)), py.values, color="#C44E52", marker="o", lw=1.6, label="papers")
ax2.set_ylabel("papers", color="#C44E52"); ax2.spines["top"].set_visible(False)
for i, n in enumerate(fy.values): ax.text(i, n + 200, f"{n:,}", ha="center", fontsize=7)
ax.set_ylabel("findings", color="#4C72B0"); ax.set_xlabel("publication year")
ax.set_ylim(0, fy.max()*1.18)
ax.set_title("Findings and papers by year — the recent skew is corpus\nrecency bias, not a surge in research volume", loc="left")
fig.tight_layout(); fig.savefig("fig09_by_year.png")

## Figure 10 — Fill-rate profile

The transparency figure: which fields you can rely on, and which are thin enrichment.

In [21]:
def fill_rates(df):
    out = []
    for c in df.columns:
        s = df[c]
        nn = (s.notna() & (s.astype(str).str.strip() != "")) if s.dtype == object else s.notna()
        out.append((c, round(100 * nn.sum() / len(df), 2)))
    return pd.DataFrame(out, columns=["field","fill_pct"]).set_index("field").sort_values("fill_pct")
FR = fill_rates(F)
KEY = ["finding_text","direction","condition","evidence_tier","provenance_quote_trunc","pathway_or_mechanism",
       "quantitative_detail","demographics","n_participants","symptom_correlate","entity","effect_size",
       "canonical_entity","treatment_class","validated_instrument","entity_ontology_id","canonical_ontology_id",
       "contradicts_finding_ref","phenotype_hpo","dataset_accession","pathway_id"]
show("Fig 10 — fill rates (%)", {k: FR.loc[k,"fill_pct"] for k in KEY})

--- Fig 10 — fill rates (%) ---
  finding_text                                      100.0
  direction                                         100.0
  condition                                         100.0
  evidence_tier                                     100.0
  provenance_quote_trunc                            100.0
  pathway_or_mechanism                               63.9
  quantitative_detail                               63.16
  demographics                                      56.58
  n_participants                                    53.86
  symptom_correlate                                 53.15
  entity                                            52.96
  effect_size                                       37.98
  canonical_entity                                  29.78
  treatment_class                                   21.66
  validated_instrument                               16.3
  entity_ontology_id                                12.67
  canonical_ontology_id                 

In [22]:
sub = FR.loc[KEY].sort_values("fill_pct")
def band(v): return "#C44E52" if v < 5 else ("#DD8452" if v < 15 else ("#55A868" if v < 60 else "#4C72B0"))
fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.barh(range(len(sub)), sub.fill_pct.values, color=[band(v) for v in sub.fill_pct.values])
for i, v in enumerate(sub.fill_pct.values): ax.text(v + 1.2, i, f"{v}%", va="center", fontsize=7.5)
ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.index, fontsize=8)
ax.set_xlim(0, 118); ax.set_xlabel("% of 36,459 rows filled (measured)")
ax.axvline(15, color="#888888", ls=":", lw=1)
ax.text(16, -0.7, "below 15%: filter, never a key", fontsize=7, color="#666666")
ax.set_title("Measured field fill rates — red fields are thin because the papers\ndon't state them, not because extraction failed", loc="left")
fig.tight_layout(); fig.savefig("fig10_fill_rates.png")

## All figures reproduced

`fig01_scope` · `fig02_conditions` · `fig03_evidence` · `fig04_entity_coverage` ·
`fig05_top_entities` · `fig06_direction` · `fig07_discovery_tables` · `fig08_nulls` ·
`fig09_by_year` · `fig10_fill_rates`

Every number printed above comes from the shipped parquet files. If a published chart disagrees
with what this notebook prints, the chart is wrong.

In [23]:
import glob
print("figures written:"); [print("  ", x) for x in sorted(glob.glob("fig*.png"))]

figures written:
   fig01_scope.png
   fig02_conditions.png
   fig03_evidence.png
   fig04_entity_coverage.png
   fig05_top_entities.png
   fig06_direction.png
   fig07_discovery_tables.png
   fig08_nulls.png
   fig09_by_year.png
   fig10_fill_rates.png
